# Integrated Drug Discovery Pipeline

This notebook demonstrates a complete, end-to-end drug target profiling workflow
using **BioServices** to orchestrate multiple bioinformatics web services.

## Pipeline overview

```
Start: Chemical structure / compound ID (ChEMBL)
  ↓  Map to targets (ChEMBL)  →  UniProt IDs
  ↓  Retrieve protein sequences (UniProt)
  ↓  Find 3D structures (PDB)
  ↓  Check protein interactions (IntAct)
  ↓  Map to pathways (Reactome / KEGG)
Output: Complete drug target profile
```

We use **Imatinib** (brand name Gleevec / Glivec, ChEMBL ID `CHEMBL941`) as the
worked example.  Imatinib is a kinase inhibitor approved for treating several
cancers and has well-characterised targets, making it ideal for demonstrating
the pipeline.


In [ ]:
import pandas as pd

from bioservices import ChEMBL, UniProt, PDB, IntactComplex, Reactome, KEGG


## Step 1 – Select the compound (ChEMBL)

[ChEMBL](https://www.ebi.ac.uk/chembl/) is a manually curated database of
bioactive drug-like small molecules.  We start by searching for Imatinib to
confirm its ChEMBL identifier and retrieve basic chemical properties.


In [ ]:
chembl = ChEMBL(verbose=False)

# Search by synonym to confirm the ChEMBL ID
search_results = chembl.search_molecule("imatinib")
hits = search_results.get("molecules", [])
print(f"Found {len(hits)} molecule(s) matching 'imatinib'")
for hit in hits[:3]:
    print(f"  {hit['molecule_chembl_id']:15s}  {hit.get('pref_name', 'N/A')}")


In [ ]:
# Retrieve full details for Imatinib
COMPOUND_ID = "CHEMBL941"

mol_data = chembl.get_molecule(COMPOUND_ID)
# get_molecule returns a list when called with a single ID
if isinstance(mol_data, list):
    mol = mol_data[0]
else:
    mol = mol_data

props = mol.get("molecule_properties") or {}
print(f"Compound      : {mol.get('pref_name')}")
print(f"ChEMBL ID     : {mol.get('molecule_chembl_id')}")
print(f"Max phase     : {mol.get('max_phase')}  (4 = approved drug)")
print(f"Mol. formula  : {props.get('molecular_formula')}")
print(f"Mol. weight   : {props.get('full_mwt')}")
print(f"RO5 violations: {props.get('num_ro5_violations')}")


## Step 2 – Discover drug targets (ChEMBL → UniProt IDs)

ChEMBL links compounds to measured bioactivities (IC50, Ki, …) against
specific protein targets.  We retrieve bioactivity records for Imatinib, then
resolve each ChEMBL target to its UniProt accession(s).


In [ ]:
# Retrieve bioactivity records for the compound
activities = chembl.get_activity(
    filters=f"molecule_chembl_id={COMPOUND_ID}&target_type=SINGLE PROTEIN",
    limit=100,
)

print(f"Retrieved {len(activities)} activity records")

# Collect unique target ChEMBL IDs with high-confidence data (pchembl_value present)
target_ids = {
    act["target_chembl_id"]
    for act in activities
    if act.get("pchembl_value") and act.get("target_chembl_id")
}
print(f"Unique high-confidence targets: {len(target_ids)}")
print("Target ChEMBL IDs:", sorted(target_ids)[:10])


In [ ]:
# Resolve ChEMBL target IDs to UniProt accessions
uniprot_accessions = {}

for target_id in sorted(target_ids):
    target_data = chembl.get_target(target_id)
    if isinstance(target_data, list):
        target = target_data[0] if target_data else {}
    else:
        target = target_data or {}

    components = target.get("target_components", [])
    for comp in components:
        acc = comp.get("accession")
        if acc:
            uniprot_accessions[acc] = {
                "target_chembl_id": target_id,
                "target_name": target.get("pref_name", "N/A"),
                "target_type": target.get("target_type", "N/A"),
            }

print(f"UniProt accessions resolved: {len(uniprot_accessions)}")
for acc, info in list(uniprot_accessions.items())[:8]:
    print(f"  {acc}  →  {info['target_name']}")


## Step 3 – Retrieve protein sequences (UniProt)

[UniProt](https://www.uniprot.org/) is the authoritative resource for protein
sequence and functional information.  We fetch the canonical entry and FASTA
sequence for each target protein identified in the previous step.

Four primary targets of Imatinib are used as a focused example:

| Protein | UniProt | Description |
|---------|---------|-------------|
| BCR-ABL1 | P00519 | Tyrosine-protein kinase ABL1 |
| KIT | P10721 | Mast/stem cell growth factor receptor |
| PDGFRA | P16234 | Platelet-derived growth factor receptor α |
| PDGFRB | P09619 | Platelet-derived growth factor receptor β |


In [ ]:
uniprot = UniProt(verbose=False)

# Focus on the four canonical Imatinib targets
PRIMARY_TARGETS = ["P00519", "P10721", "P16234", "P09619"]

protein_info = {}
for acc in PRIMARY_TARGETS:
    entry = uniprot.retrieve(acc, frmt="json")
    if not entry:
        continue

    gene_names = entry.get("genes", [{}])
    gene = gene_names[0].get("geneName", {}).get("value", "N/A") if gene_names else "N/A"
    seq_info = entry.get("sequence", {})

    protein_info[acc] = {
        "gene": gene,
        "protein_name": entry.get("proteinDescription", {}).get("recommendedName", {}).get("fullName", {}).get("value", "N/A"),
        "organism": entry.get("organism", {}).get("scientificName", "N/A"),
        "length": seq_info.get("length", 0),
        "reviewed": entry.get("entryType", "").startswith("UniProtKB reviewed"),
    }

df_proteins = pd.DataFrame(protein_info).T
df_proteins.index.name = "UniProt"
print(df_proteins.to_string())


In [ ]:
# Retrieve FASTA sequences for all primary targets
fasta_sequences = {}
for acc in PRIMARY_TARGETS:
    fasta = uniprot.get_fasta(acc)
    if fasta:
        # Store the sequence (everything after the header line)
        lines = fasta.strip().splitlines()
        header = lines[0]
        sequence = "".join(lines[1:])
        fasta_sequences[acc] = sequence
        print(f"{acc}: {header[:60]}")
        print(f"      Sequence length = {len(sequence)} aa")


## Step 4 – Find 3D structures (PDB)

The [Protein Data Bank (PDB)](https://www.rcsb.org/) contains experimentally
determined 3D structures.  We query for structures of each target protein,
optionally filtering by resolution or experimental method.


In [ ]:
pdb = PDB()

pdb_hits = {}
for acc in PRIMARY_TARGETS:
    # Search PDB for entries containing this UniProt accession
    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                "operator": "exact_match",
                "value": acc,
            },
        },
        "request_options": {"paginate": {"start": 0, "rows": 10}},
        "return_type": "entry",
    }
    result = pdb.search(query)
    ids = []
    if result and isinstance(result, dict):
        ids = [r["identifier"] for r in result.get("result_set", [])]
    pdb_hits[acc] = ids
    gene = protein_info.get(acc, {}).get("gene", acc)
    print(f"{acc} ({gene}): {len(ids)} PDB structure(s)  →  {ids[:5]}")


## Step 5 – Explore protein interactions (IntAct)

The [IntAct Complex Portal](https://www.ebi.ac.uk/intact/complex/) provides
manually curated macromolecular complexes and protein–protein interactions.
We search for known complexes involving each target protein.


In [ ]:
intact = IntactComplex(verbose=False)

interaction_summary = {}
for acc in PRIMARY_TARGETS:
    gene = protein_info.get(acc, {}).get("gene", acc)
    result = intact.search(
        gene,
        filters='species_f:("Homo sapiens")',
        number=10,
    )
    elements = result.get("elements", []) if isinstance(result, dict) else []
    interaction_summary[acc] = [
        {
            "complex_id": el.get("complexAC"),
            "name": el.get("complexName"),
        }
        for el in elements[:5]
    ]
    print(f"{acc} ({gene}): {len(elements)} complex record(s)")
    for entry in interaction_summary[acc]:
        print(f"    {entry['complex_id']}  {entry['name']}")


## Step 6 – Map to biological pathways (Reactome & KEGG)

### Reactome

[Reactome](https://reactome.org/) is a curated pathway and reaction knowledgebase.
We map each UniProt accession to the pathways it participates in.


In [ ]:
reactome = Reactome(verbose=False)

reactome_pathways = {}
for acc in PRIMARY_TARGETS:
    pathways = reactome.get_mapping_identifier_pathways("UniProt", acc)
    if isinstance(pathways, list):
        reactome_pathways[acc] = [
            {"id": p.get("stId"), "name": p.get("displayName")}
            for p in pathways[:5]
        ]
    else:
        reactome_pathways[acc] = []

    gene = protein_info.get(acc, {}).get("gene", acc)
    print(f"{acc} ({gene}): {len(reactome_pathways[acc])} pathway(s) (showing up to 5)")
    for pw in reactome_pathways[acc]:
        print(f"    {pw['id']}  {pw['name']}")


### KEGG

[KEGG](https://www.genome.jp/kegg/) (Kyoto Encyclopedia of Genes and Genomes)
provides metabolic and signalling pathway maps.  We convert UniProt IDs to KEGG
gene identifiers, then retrieve the associated pathway IDs.


In [ ]:
kegg = KEGG(verbose=False)

kegg_pathways = {}
for acc in PRIMARY_TARGETS:
    gene = protein_info.get(acc, {}).get("gene", acc)

    # Convert UniProt accession to KEGG human gene ID
    conv_result = kegg.conv("hsa", f"uniprot:{acc}")
    if not conv_result:
        print(f"{acc} ({gene}): no KEGG mapping found")
        kegg_pathways[acc] = []
        continue

    # conv_result maps 'up:ACC' → 'hsa:ENTREZ_ID'
    kegg_gene_id = next(iter(conv_result.values()))  # e.g. 'hsa:25'
    numeric_id = kegg_gene_id.split(":")[1]          # e.g. '25'

    # Find pathways that contain this gene
    pathways = kegg.get_pathway_by_gene(numeric_id, "hsa")
    kegg_pathways[acc] = pathways[:5] if pathways else []

    print(f"{acc} ({gene}) → KEGG ID {kegg_gene_id}: {len(pathways) if pathways else 0} pathway(s)")
    for pw in kegg_pathways[acc]:
        print(f"    {pw}")


## Output – Complete drug target profile

We now assemble all the information gathered across the pipeline into a single
summary table – one row per primary target protein.


In [ ]:
rows = []
for acc in PRIMARY_TARGETS:
    info = protein_info.get(acc, {})
    rows.append(
        {
            "UniProt": acc,
            "Gene": info.get("gene", "N/A"),
            "Protein name": info.get("protein_name", "N/A"),
            "Length (aa)": info.get("length", 0),
            "PDB structures": len(pdb_hits.get(acc, [])),
            "IntAct complexes": len(interaction_summary.get(acc, [])),
            "Reactome pathways": len(reactome_pathways.get(acc, [])),
            "KEGG pathways": len(kegg_pathways.get(acc, [])),
        }
    )

df_profile = pd.DataFrame(rows).set_index("UniProt")
print(f"\n{'='*70}")
print(f"  Complete drug target profile for {COMPOUND_ID} (Imatinib)")
print(f"{'='*70}")
print(df_profile.to_string())


In [ ]:
# Display detailed pathway listings for each target
print("\nReactome pathways per target:")
for acc in PRIMARY_TARGETS:
    gene = protein_info.get(acc, {}).get("gene", acc)
    print(f"\n  {acc} ({gene}):")
    for pw in reactome_pathways.get(acc, []):
        print(f"    [{pw['id']}] {pw['name']}")

print("\nKEGG pathways per target:")
for acc in PRIMARY_TARGETS:
    gene = protein_info.get(acc, {}).get("gene", acc)
    print(f"\n  {acc} ({gene}):")
    for pw in kegg_pathways.get(acc, []):
        print(f"    {pw}")
